# HuggingFace coverage over time (Wayback Machine)

Reconstructs historical per-language dataset and model counts from archived
HuggingFace snapshots retrieved via the Wayback Machine.

In [ ]:
import requests
from bs4 import BeautifulSoup
from collections import defaultdict
import pandas as pd
import time
from datetime import datetime

In [ ]:
# Step 1: Get one snapshot per month using the CDX API
def get_monthly_snapshots(domain):
    cdx_url = f"http://web.archive.org/cdx/search/cdx?url={domain}&output=json&fl=timestamp,original&collapse=timestamp:8"
    response = requests.get(cdx_url)
    response.raise_for_status()
    data = response.json()
    snapshots = data[1:]  # skip header
    return snapshots

def safe_int(text):
    text = text.strip().replace(",", "")
    return int(text) if text.isdigit() else None


# Step 2: Parse the language table from each snapshot
def parse_language_table(archive_url, snapshot_date):
    try:
        res = requests.get(archive_url, timeout=10)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, 'html.parser')
        rows = soup.find_all("tr")

        records = []
        for row in rows:
            cols = row.find_all("td")
            if len(cols) < 4:
                continue

            lang_name = cols[0].get_text(strip=True).split("\n")[0]
            code_tag = cols[1].find("code")
            dataset_tag = cols[2].find("a")
            model_tag = cols[3].find("a")

            if code_tag and dataset_tag and model_tag:
                code = code_tag.text.strip()
                datasets = safe_int(dataset_tag.text)
                models = safe_int(model_tag.text)   
                records.append({
                    "date": snapshot_date,
                    "language": lang_name,
                    "code": code,
                    "datasets": datasets,
                    "models": models
                })
        print(records)
        return records
    except Exception as e:
        print(f"Failed to process {archive_url}: {e}")
        return []

# Step 3: Main routine
def scrape_huggingface_languages():
    domain = "huggingface.co/languages"
    snapshots = get_monthly_snapshots(domain)

    all_data = []
    for i, (timestamp, original_url) in enumerate(snapshots):
        if i > 0 and i % 10 == 0:
            print("Pausing for 30 seconds to be polite to Wayback Machine...")
            time.sleep(30)

        snapshot_date = datetime.strptime(timestamp, "%Y%m%d%H%M%S").strftime("%Y-%m")
        archive_url = f"https://web.archive.org/web/{timestamp}/{original_url}"
        print(f"Processing snapshot {i+1}/{len(snapshots)}: {snapshot_date}")
        records = parse_language_table(archive_url, snapshot_date)
        all_data.extend(records)
        time.sleep(5)
        
    df = pd.DataFrame(all_data)
    return df

# Run it
df = scrape_huggingface_languages()
df.to_csv("huggingface_languages_timeseries.csv", index=False)
print("Saved to huggingface_languages_timeseries.csv")